# LegalIR Task 1: Google Colab A100 Production Training (B1.2)
## UIT Data Science Challenge 2026 — High-Recall Vietnamese Legal IR
**Pinned Git Commit:** `3743464c20835faeb0c62c3a292c1c7d0e7202d6`

### Production Training Invariants:
- **Enforces NVIDIA A100 GPU** before consuming compute credits.
- Verifies prior Kaggle Dual-T4 report and Colab Single-T4 report.
- Trains `BAAI/bge-reranker-v2-m3` LoRA on all 7,000 canonical training queries.
- Uses `torch.bfloat16` precision end-to-end.
- Generates Top-5 predictions for 1,000 official public test queries.
- Verifies all submission invariants and builds `submission.zip`.
- Captures immutable Hugging Face release revision into `run_manifest.json`.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Verification & Local Environment Loader
# Fail-closed A100 enforcement BEFORE burning compute credits.
# ==============================================================================
import json
import os
import sys
import torch
from pathlib import Path

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU required for training."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[+] Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
assert "A100" in gpu_name, f"A100 required (found {gpu_name}). Select Runtime -> Change runtime type -> A100."

# Prefer Colab Secrets, fallback to uploaded .env (CLI automation)
try:
    from google.colab import userdata
    _ud = userdata.get
except Exception:
    _ud = None
if _ud is not None:
    for _k in ["HF_TOKEN", "HF_TOKEN_WRITE", "HF_TOKEN_READ", "KAGGLE_API_TOKEN", "KAGGLE_KEY", "HF_REPO_ID"]:
        try:
            _v = _ud(_k)
        except Exception:
            _v = None
        if _v and _k not in os.environ:
            os.environ[_k] = str(_v)

for env_path in [Path("/content/.env"), Path("/content/LegalIR/.env"), Path(".env")]:
    if env_path.is_file():
        print(f"[+] Loading local environment from {env_path}...")
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip("'\""))

# Normalize HF Token (fine-grained or classic; accept HF_TOKEN_WRITE, HF_TOKEN, HF_TOKEN_READ; ignore non-HF tokens like KGAT_)
# Required scopes for release: read access to public models + write access to HF_REPO_ID.
hf_candidates = [os.environ.get("HF_TOKEN_WRITE"), os.environ.get("HF_TOKEN"), os.environ.get("HF_TOKEN_READ")]
hf_tok = next((t for t in hf_candidates if t and str(t).startswith("hf_")), None)
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
    try:
        from huggingface_hub import HfApi
        u_name = HfApi(token=hf_tok).whoami().get("name", "unknown")
        print(f"[+] HF_TOKEN verified (authenticated as @{u_name}).")
        print("    • Model Downloads: Authenticated (rate-limit bypass & priority CDN)")
        print("    • Model Uploads  : Attempted for private repo", os.environ.get("HF_REPO_ID", "dangphuc2109/legalir-task1-reranker"), "(write scope required; verified pre-training)")
    except Exception as exc:
        print(f"[+] HF_TOKEN active in environment for model downloads & uploads ({exc}).")
else:
    print("[!] Notice: Valid HF_TOKEN not found. Model downloads will be anonymous; uploads disabled.")

# Normalize Kaggle credentials for dataset acquisition
kg_tok = os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY")
if kg_tok and str(kg_tok).startswith("KGAT_"):
    os.environ["KAGGLE_API_TOKEN"] = kg_tok
    os.environ["KAGGLE_KEY"] = kg_tok
    kg_cfg = Path.home() / ".kaggle" / "kaggle.json"
    kg_cfg.parent.mkdir(parents=True, exist_ok=True)
    kg_user = os.environ.get("KAGGLE_USERNAME", "phucdangg")
    kg_cfg.write_text(json.dumps({"username": kg_user, "key": kg_tok}))
    kg_cfg.chmod(0o600)
    print("[+] Kaggle CLI configured from local credentials.")


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Detached HEAD Checkout
# ==============================================================================
import subprocess
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA") or "3743464c20835faeb0c62c3a292c1c7d0e7202d6"
REPO_DIR = Path("/content/LegalIR") if Path("/content").exists() else Path.cwd()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", EXPECTED_COMMIT], cwd=REPO_DIR, check=False)
    print(f"[*] Checking out exact commit: {EXPECTED_COMMIT} (detached HEAD)...")
    res = subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[*] Checkout fallback: unshallowing repository...")
        subprocess.run(["git", "fetch", "--unshallow", "origin"], cwd=REPO_DIR, check=False)
        subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[+] Working in: {REPO_DIR}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies & Canonical Dataset Setup
# Do NOT reinstall torch (Colab CUDA build). Install only missing wheels.
# ==============================================================================
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "pyvi", "pyarrow", "huggingface_hub", "bm25s", "faiss-cpu", "lightgbm", "scikit-learn", "sentencepiece"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
sys.modules["torchao"] = None
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

dataset_dir = Path("/content/kaggle_dataset") if Path("/content").exists() else REPO_DIR / "artifacts/shared/canonical/v2"
required_files = ["documents.parquet", "chunks.parquet", "queries_train.parquet", "qrels_train.parquet", "public-official.json"]
if not all((dataset_dir / _f).is_file() for _f in required_files):
    dataset_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "phucdangg/legalir-task1-clean-data", "-p", str(dataset_dir), "--unzip", "--force"], check=True)
    missing = [_f for _f in required_files if not (dataset_dir / _f).is_file()]
    assert not missing, f"Dataset incomplete after download, missing: {missing}"
print(f"[+] Dataset verified at: {dataset_dir}")


In [ ]:
# ==============================================================================
# Cell 4: Execute Colab A100 Production Gate (scripts/run_colab_train.py -> scripts/gates/run_a100.py)
# CLI equivalent: python scripts/run_colab_train.py --dataset-dir /content/kaggle_dataset --output-dir /content/legalir_production_run --expected-sha $EXPECTED_COMMIT
# ==============================================================================
from scripts.run_colab_train import run_colab_production_training

output_dir = Path("/content/legalir_production_run") if Path("/content").exists() else REPO_DIR / "artifacts/task1/production"
output_dir.mkdir(parents=True, exist_ok=True)

hf_repo = os.environ.get("HF_REPO_ID", "dangphuc2109/legalir-task1-reranker")
k_cands = [Path("/content/kaggle_t4x2_report.json"), REPO_DIR / "artifacts/task1/gates/kaggle_t4x2_report.json"]
k_report_p = next((p for p in k_cands if p.is_file()), None)
c_cands = [Path("/content/colab_t4_report.json"), REPO_DIR / "artifacts/task1/gates/colab_t4_report.json"]
c_report_p = next((p for p in c_cands if p.is_file()), None)
freeze_cands = [Path("/content/production_freeze.json"), REPO_DIR / "artifacts/task1/freeze/production_freeze.json"]
freeze_p = next((p for p in freeze_cands if p.is_file()), None)

report = run_colab_production_training(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    smoke_report_path=k_report_p,
    colab_t4_report_path=c_report_p,
    expected_sha=EXPECTED_COMMIT,
    precision="bf16",
    allow_non_a100=False,
    mock=False,
    hf_repo=hf_repo,
    freeze_file_path=freeze_p,
    run_mode="full",
)
print(f"[+] A100 Production Gate execution status: {report.get('status')} | Verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Verify Submission & Report Release State
# ==============================================================================
from src.evaluation.submission import validate_submission_zip

sub_zip = output_dir / "submission.zip"
zip_val = validate_submission_zip(sub_zip)
assert zip_val.get("is_valid"), f"Submission validation failed: {zip_val.get('errors')}"
print(f"[+] SUCCESS: submission.zip validated cleanly at {sub_zip}")

manifest_p = output_dir / "run_manifest.json"
assert manifest_p.is_file(), f"Run manifest missing at {manifest_p}"
m_data = json.loads(manifest_p.read_text(encoding='utf-8'))
print(f"[+] Run Status : {m_data.get('status')}")
print(f"[+] Verdict    : {m_data.get('verdict')}")
if m_data.get("huggingface"):
    hf_meta = m_data["huggingface"]
    print(f"[+] Hugging Face Release: https://huggingface.co/{hf_meta.get('repo_id')}")
    print(f"    Release Commit      : {hf_meta.get('commit_sha')}")
